# 07 · Graph Neural Networks

Represent a customer–product bipartite graph, normalize sparse adjacency, implement message passing, train a manual PyTorch GCN, compare node/link tasks with traditional baselines, and retain negative results.

This is a learning and interview notebook, not a thin launcher. It combines mechanism,
executable evidence, diagnostics, limitations, and experiments. Run from top to bottom;
keep `RUN_TRAINING=False` to inspect committed/CI results, or turn it on to reproduce them.

In [ ]:
# Local/VS Code and Colab bootstrap. No secret or API key is used.
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    repo = Path("/content/ai-rookie-skill-tree-lab")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/chelsietao/ai-rookie-skill-tree-lab.git", str(repo)],
            check=True,
        )
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[full,dev]"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

ROOT = Path.cwd().resolve()
DATA_DIR = ROOT / "data/generated/ci"
METRICS_DIR = ROOT / "artifacts/metrics/ci"
RUN_TRAINING = False  # Set True when you want to regenerate the chapter's metrics.
assert (ROOT / "pyproject.toml").exists(), "Open the repository root or notebooks directory."
print({"environment": "Colab" if IN_COLAB else "local/VS Code", "root": str(ROOT)})

## Learning objectives

- Map external IDs to contiguous node indices.
- Build undirected sparse adjacency with self-loops and symmetric normalization.
- Explain graph convolution as neighbor aggregation plus a learned linear map.
- Separate node classification from link prediction evaluation.
- Compare GNN scores against degree/tabular baselines before claiming graph value.

In [ ]:
if not (DATA_DIR / "manifest.json").exists():
    subprocess.run(
        [sys.executable, "scripts/prepare_data.py", "--profile", "ci", "--output", str(DATA_DIR)],
        check=True,
    )
subprocess.run([sys.executable, "scripts/validate_data.py", "--data-dir", str(DATA_DIR)], check=True)
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print(manifest["dataset_version"], manifest["rows"])

In [ ]:
import pandas as pd
edges = pd.read_csv(DATA_DIR / "graph_edges.csv.gz")
customers = pd.read_csv(DATA_DIR / "customers.csv.gz")
display(edges.head())
print({"customers": customers.customer_id.nunique(), "products": edges.product_id.nunique(), "edges": len(edges)})

## Bipartite representation

Edges connect customer nodes only to product nodes. There are no customer–customer or product–product
edges in the raw graph. Transactions and browse events both contribute unique relationships.

In [ ]:
customer_degree = edges.groupby("customer_id").size()
product_degree = edges.groupby("product_id").size()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
customer_degree.hist(ax=axes[0], bins=30); axes[0].set_title("Customer degree")
product_degree.hist(ax=axes[1], bins=30); axes[1].set_title("Product degree")
plt.tight_layout(); plt.show()

## Normalized adjacency

The GCN uses $\hat A=D^{-1/2}(A+I)D^{-1/2}$. Self-loops retain a node's representation; symmetric
degree normalization prevents high-degree neighbors from dominating every update.

In [ ]:
import torch
from rookie_lab.graph.gcn import normalized_adjacency

tiny_edges = torch.tensor([[0, 1], [2, 2]])
tiny_adjacency = normalized_adjacency(tiny_edges, 3)
display(pd.DataFrame(tiny_adjacency.to_dense().numpy()).round(3))
assert torch.allclose(tiny_adjacency.to_dense(), tiny_adjacency.to_dense().T)

## Message passing by hand

One graph-convolution layer computes normalized neighbor aggregation and then a linear projection.
Sparse multiplication avoids materializing the full adjacency for larger graphs.

In [ ]:
features = torch.eye(3)
aggregated = torch.sparse.mm(tiny_adjacency, features)
display(pd.DataFrame(aggregated.numpy()).round(3))

In [ ]:
metric_path = METRICS_DIR / "graph.json"
if RUN_TRAINING or not metric_path.exists():
    metric_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable, "scripts/run_graph.py", "--data-dir", str(DATA_DIR), "--profile", "ci", "--output", str(METRICS_DIR / "graph.json")], check=True)
result = json.loads(metric_path.read_text())

print("loaded", metric_path, "dataset", result.get("_run", {}).get("dataset_version"))

In [ ]:
print(result["graph_statistics"])
print(result["implementation"])

## Node classification

Customer features occupy the first six dimensions; product nodes receive synthetic features. A
two-layer GCN predicts customer churn. The comparison baseline uses the same tabular customer
features plus node degree in class-weighted logistic regression.

In [ ]:
node_comparison = pd.Series({
    "GCN": result["node_classification"]["roc_auc"],
    "Logistic + degree": result["traditional_baseline"]["node_roc_auc"],
})
display(node_comparison.rename("node_roc_auc").to_frame())
node_comparison.plot.bar(title="Does message passing beat the baseline?"); plt.ylim(0, 1); plt.show()

## Link prediction and negative sampling

Positive links are observed customer–product pairs. Negatives are sampled from absent pairs. The
GCN scores an edge with the dot product of endpoint embeddings. Sampling policy changes task
difficulty and must be documented.

In [ ]:
display(pd.Series(result["link_prediction"], name="roc_auc").to_frame())

## Honest negative result

In the quick run, the GCN link score can be below the degree-popularity baseline. That means the
current product features, graph generator, objective, or training budget does not justify claiming
superior graph link prediction. Keeping this result is stronger research evidence than hiding it.

In [ ]:
loss = result["training_loss"]["curve"]
plt.plot(loss); plt.title("Joint node + link reconstruction loss"); plt.xlabel("epoch"); plt.show()
print("loss change", loss[-1] - loss[0])

## Transductive limitation

All nodes are present while training, even though only training customer labels are used. This is
transductive learning. A true new-customer/product deployment needs inductive features and an
evaluation split that withholds nodes, not only labels or edges.

In [ ]:
print({"scope": result["scope"], "nodes": result["nodes"], "edges": result["edges"]})

## Exercises and interview checkpoint

1. Use hard negatives with popular products rather than uniform negatives.
2. Hold out entire customers and explain the inductive changes required.
3. Add product category features and test link ROC-AUC.
4. Compare one versus two graph-convolution layers for oversmoothing.
5. State why a falling joint loss does not guarantee better node and link metrics.